[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SeyhaLite/ml-foundations/blob/main/docs/notebooks/06_inference_and_export.ipynb)

# ០៦ · Inference, Merge និង Export

In [ ]:
# ⚙️ Setup — រត់ cell នេះមុនគេ (Auto-setup for Google Colab)
import os, subprocess, sys

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir("ml-foundations"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/SeyhaLite/ml-foundations.git"], check=True)
    os.chdir("ml-foundations")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# ធ្វើឱ្យ qwen_ft import បានភ្លាមៗក្នុង session នេះ (មិនបាច់ restart)
for _cand in ("src", os.path.join("..", "..", "src")):
    _cand = os.path.abspath(_cand)
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import qwen_ft
print("qwen-ft", qwen_ft.__version__, "ready ✅")

## Sampling presets ផ្លូវការសម្រាប់ Qwen3.5

In [ ]:
from qwen_ft.inference import SAMPLING_PRESETS
for name, params in SAMPLING_PRESETS.items():
    print(f"{name:22s} {params}")

## ផ្ទុក base + adapter → សន្ទនា (guarded)

```bash
export QWEN_FT_ALLOW_HEAVY=1
```

បើអ្នកបានហ្វឹក adapter ក្នុង notebook ០៥ — ប្រើ path នោះ។
បើអត់ — សាកល្បងជាមួយ base model ធម្មតា។

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # សន្ទនាជាមួយម៉ូដែល (base ឬ +adapter)
    from qwen_ft.inference import load_for_chat, chat

    ADAPTER = "outputs/qwen3-0.6b-khmer/adapter"   # កែតាម adapter របស់អ្នក
    import os
    adapter = ADAPTER if os.path.isdir(ADAPTER) else None

    model, tokenizer = load_for_chat("Qwen/Qwen3-0.6B", adapter_path=adapter)
    print("adapter loaded:", adapter or "(base only)")

    for q in ["សួស្តី! ណែនាំខ្លួនឯងមក", "តើអ្វីជា fine-tuning?"]:
        reply = chat(model, tokenizer, [{"role": "user", "content": q}],
                     max_new_tokens=128, enable_thinking=False, preset="instruct_general")
        print(f"Q: {q}\nA: {reply}\n")
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## Merge adapter → ម៉ូដែលឯករាជ្យ (guarded)

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # merge LoRA ចូលទម្ងន់ base
    from qwen_ft.inference import merge_adapter

    merged_dir = merge_adapter(
        model_id_or_path="Qwen/Qwen3-0.6B",
        adapter_path="outputs/qwen3-0.6b-khmer/adapter",
        out_dir="outputs/qwen3-0.6b-khmer-merged",
        dtype="bfloat16",
    )
    print("merged model:", merged_dir)
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## Export ទៅ GGUF + Deploy (មិនរត់ក្នុង notebook)

**GGUF (Ollama/llama.cpp)៖** មើលជំពូក
[៨ · រក្សាទុក · Merge · Deploy](../chapters/ch08-save-merge-deploy.md) —
ការ convert ត្រូវការ llama.cpp build ដោយផ្ទាល់។

**vLLM (production)៖**
```bash
vllm serve outputs/qwen3-0.6b-khmer-merged --max-model-len 8192
```

### លំហាត់
1. ប្រៀបធៀបចម្លើយ base vs +adapter លើសំណួរដូចគ្នា (greedy)
2. ផ្ទៀងផ្ទាត់ថា merge ត្រឹមត្រូវ — ចម្លើយ greedy ត្រូវដូចគ្នាបេះបិទ
3. សរសេរ Modelfile សម្រាប់ Ollama ជាមួយ system prompt ខ្មែរ